Comparo [[Naive Bayes#^3fb98d|naive bayes gaussiano]] e la sua versione discrera nel caso _mnist_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# Load dataset
from tensorflow.keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train / 255.
x_test = x_test / 255.

print("min value = {}, max value = {}".format(np.min(x_train), np.max(x_train)))
print(x_train.shape)
print(x_test.shape)


In [ ]:
def show_samples(samples):
    n = np.shape(samples)[0]
    plt.figure(figsize=(2 * n, 4))
    for i in range(n):
        ax = plt.subplot(1, n, i + 1)
        plt.imshow(samples[i])
        plt.gray()
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
    plt.show()

show_samples(x_test[20:30])


In [ ]:
mean_digits = np.zeros((10, 28, 28))
for i in range(0, 10):
    mean_digits[i] = np.mean(x_train[y_train == i], axis=0)
show_samples(mean_digits)


In [ ]:
# Compute the cardinality of all categories
card = np.zeros((10,))
for i in range(0, 10):
    card[i] = np.sum(y_train == i)

print("card = ", card)


We can also plot these frequencies as a histogram.


In [ ]:
def plot_hist(a, bins=10, title=None):
    plt.figure(tight_layout=True)
    plt.hist(a, bins=bins)
    if title:
        plt.title(title)
    plt.show()

plot_hist(y_train, title="Data for categories")


Ora si calcolano i parametri per naive bayes gaussiano 
Si ignorano le priori $P(Y)$ visto che il dataset è sufficientemente bilanciato
Per ogni categoria e pixel, si calcola la [[Media]] e la [[Varianza]]

In [ ]:
Mean = np.zeros((10, 28, 28))
Var = np.zeros((10, 28, 28))
epsilon = 1e-5  # Small positive value

for i in range(0, 10):
    Mean[i] = np.mean(x_train[y_train == i], axis=0)
    Var[i] = np.var(x_train[y_train == i], axis=0)
    Var[i] = np.maximum(Var[i], epsilon)


Ora si calcola, per ogni categoria, una distribuzione gaussiana multimodale per i pixel

In [ ]:
def gaussian_density(x, mean, var):
    numerator = np.exp(- (x - mean) ** 2 / (2 * var))
    denominator = np.sqrt(2 * np.pi * var)
    return numerator / denominator


In [ ]:
def classify(sample):
    probs = []
    for i in range(0, 10):
        pixel_probs = gaussian_density(sample, Mean[i], Var[i])
        prob = np.sum(np.log(pixel_probs))
        probs.append(prob)
    return np.argmax(probs)


In [ ]:
for i in range(100):
    prediction = classify(x_test[i])
    true = y_test[i]
    print("true = {}, predicted = {}".format(true, prediction))
    if true != prediction:
        show_samples(np.expand_dims(x_test[i], axis=0))
        break
    time.sleep(2)


# Discrete case


In [ ]:
x_train_discr = x_train > 0.5
x_test_discr = x_test > 0.5

show_samples(x_test_discr[20:30])


In [ ]:
Freq = np.zeros((10, 28, 28))
for i in range(0, 10):
    Freq[i] = np.sum(x_train_discr[y_train == i], axis=0)

# Add one to avoid zero (for log computations)
Freq += 1


In [ ]:
def freq_ij_by_category(i, j):
    freq_ij = Freq[:, i, j]
    plt.bar(np.arange(freq_ij.shape[0]), freq_ij)

freq_ij_by_category(4, 10)


In [ ]:
# Probabilities to be 1 or 0
Prob1 = Freq / card[:, None, None]
Prob0 = 1 - Prob1

print("Prob1, Prob0 shape =", Prob1.shape, Prob0.shape)

# Passing to logs
logProb1 = np.log(Prob1)
logProb0 = np.log(Prob0)

assert (logProb1 <= 0).all() & (logProb0 <= 0).all()


In [ ]:
def classify(img):
    d_img = img > 0.5
    logp = np.sum(logProb1 * d_img + logProb0 * (1 - d_img), axis=(1, 2))
    return np.argmax(logp)


In [ ]:
for i in range(100):
    prediction = classify(x_test[i])
    true = y_test[i]
    print("true = {}, predicted = {}".format(true, prediction))
    if true != prediction:
        show_samples(np.expand_dims(x_test_discr[i], axis=0))
        break
    time.sleep(2)
